# IOAI — 2024 Mock Competition Basketball Shooting (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/jaredliw/ioai-tsp-2025"
CLONE = "ioai-tsp-2025"
SUBDIR = "noai-china-2024/basketball-shooting"
WORKDIR = "noai-china-2024/basketball-shooting"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Predicting the Shooting Percentage of Basketball Stars

> **NOAI China 2024 — Question 1** (APOAI 2025 Mock, Q1).

Build a small **MLP** that predicts whether a shot is made (`shot_made_flag`)
from the shot location `loc_x`, `loc_y`.

**Constraints (enforced by the official grader):** only `nn.Linear` + activations,
**at most 3 linear layers**, **at most 8 neurons per layer**, and you must build the net
directly (no `nn.Sequential`). Score = **test accuracy** (0 if the constraints are violated).

The official A/B test sets are hidden, so here we carve a **held-out validation split**
(`shot_id % 5 == 0`, 20%) from `data_train.csv` and report validation accuracy. The
**Submit** button re-runs your saved model on the same held-out split.

In [ ]:
import os, random, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Load data and make a held-out validation split

`val = (shot_id % 5 == 0)` — a fixed 20% the model never trains on (the grader uses the same rule).

In [ ]:
df = pd.read_csv('data_train.csv')
input_cols = ['loc_x', 'loc_y']   # specified by the task
is_val = (df['shot_id'] % 5 == 0)
df_tr, df_va = df[~is_val], df[is_val]
len(df_tr), len(df_va)

In [ ]:
def to_xy(d):
    X = torch.tensor(d[input_cols].to_numpy(), dtype=torch.float32)
    y = torch.tensor(d['shot_made_flag'].to_numpy(), dtype=torch.float32)
    return X, y

X_tr, y_tr = to_xy(df_tr)
X_va, y_va = to_xy(df_va)
X_tr.shape, X_va.shape

## Define the model (<=3 Linear, <=8 neurons each, no Sequential) and train

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.act = nn.Tanh()
        self.fc1 = nn.Linear(2, 8)
        self.fc2 = nn.Linear(8, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    probs = torch.sigmoid(model(X.to(device)).squeeze())
    preds = (probs >= 0.5).float().cpu()
    return (preds == y).float().mean().item()

model = MyModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)

Xtr, ytr = X_tr.to(device), y_tr.to(device)
for epoch in range(300):
    model.train()
    out = model(Xtr).squeeze()
    loss = criterion(out, ytr)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f'epoch {epoch+1:4d}  loss {loss.item():.4f}  val_acc {accuracy(model, X_va, y_va):.4f}')

In [ ]:
val_acc = accuracy(model, X_va, y_va)
print(f'Held-out validation accuracy: {val_acc:.4f}')
# Only loc_x/loc_y are allowed, so accuracy near ~0.56 is expected (shot success is weakly
# position-dependent). To push higher you would need richer features, which the rules forbid.

## Save submission (`submission.zip` = `submission_model.py` + `submission_dic.pth`)

In [ ]:
torch.save(model.state_dict(), 'submission_dic.pth')
model_code = '''import torch
import torch.nn as nn


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.act = nn.Tanh()
        self.fc1 = nn.Linear(2, 8)
        self.fc2 = nn.Linear(8, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)
        return x
'''
with open('submission_model.py', 'w') as f:
    f.write(model_code)
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('submission_model.py'); z.write('submission_dic.pth')
print('wrote submission.zip')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)